# Patrón Creacional: Singleton

## Dominio del ejemplo: sistema bancario

## Introducción
El patrón **Singleton** garantiza que una clase tenga **una sola instancia** en todo el
programa y ofrece un **punto de acceso global** a ella.

### ¿Qué problema resuelve en la banca?
En un core bancario existe una **configuración central** (tasa base del banco, moneda,
nombre de la entidad, parámetros de conexión al host central). Si cada módulo crea su
propia copia de esa configuración, podríamos terminar con **valores contradictorios**
(un módulo cobrando con una tasa y otro con otra) y con **desperdicio de recursos**
(varias conexiones al host cuando debería existir una sola). Singleton asegura que
**toda la aplicación lea y escriba la misma configuración**.

## Código *sin patrón* (el problema es evidente)
Sin Singleton, cada parte del sistema instancia su propia configuración. Al modificar
una, las demás **no se enteran**: el estado queda desincronizado.

In [1]:
class ConfiguracionBancoSinPatron:
    def __init__(self):
        # Valores por defecto del banco
        self.nombre_banco = "Banco Central"
        self.tasa_base = 0.02
        self.moneda = "COP"


# Dos modulos distintos crean su propia configuracion
config_creditos = ConfiguracionBancoSinPatron()
config_cajero = ConfiguracionBancoSinPatron()

# El area de riesgos actualiza la tasa base en SU copia
config_creditos.tasa_base = 0.05

print("Tasa vista por el modulo de creditos:", config_creditos.tasa_base)
print("Tasa vista por el modulo de cajero: ", config_cajero.tasa_base)
print("Son el mismo objeto?", config_creditos is config_cajero)
print(">> Problema: hay dos configuraciones distintas y las tasas NO coinciden.")

Tasa vista por el modulo de creditos: 0.05
Tasa vista por el modulo de cajero:  0.02
Son el mismo objeto? False
>> Problema: hay dos configuraciones distintas y las tasas NO coinciden.


### Análisis del problema
- Existen **dos objetos** de configuración (`is` devuelve `False`).
- El cambio de tasa hecho por riesgos **no se refleja** en el módulo de cajero.
- Resultado: el banco podría cobrar tasas distintas según el módulo. Inconsistencia grave.

## Código *con patrón* (problema resuelto)
Implementamos Singleton sobrescribiendo `__new__` para que **siempre** se devuelva la
misma instancia. Ahora todos los módulos comparten la **misma** configuración.

In [2]:
class ConfiguracionBanco:
    _instancia = None  # aqui se guarda la unica instancia

    def __new__(cls):
        if cls._instancia is None:
            cls._instancia = super().__new__(cls)
            # Inicializacion unica (solo la primera vez)
            cls._instancia.nombre_banco = "Banco Central"
            cls._instancia.tasa_base = 0.02
            cls._instancia.moneda = "COP"
        return cls._instancia


# Dos modulos "distintos" piden la configuracion
config_creditos = ConfiguracionBanco()
config_cajero = ConfiguracionBanco()

# Riesgos actualiza la tasa base
config_creditos.tasa_base = 0.05

print("Tasa vista por el modulo de creditos:", config_creditos.tasa_base)
print("Tasa vista por el modulo de cajero: ", config_cajero.tasa_base)
print("Son el mismo objeto?", config_creditos is config_cajero)
print(">> Solucion: una sola configuracion compartida; la tasa es consistente.")

Tasa vista por el modulo de creditos: 0.05
Tasa vista por el modulo de cajero:  0.05
Son el mismo objeto? True
>> Solucion: una sola configuracion compartida; la tasa es consistente.


### Verificación
- `config_creditos is config_cajero` ahora es **True**: es el mismo objeto.
- El cambio de tasa se ve reflejado en **ambos** módulos.
- El banco opera con una **única fuente de verdad** para su configuración.

## UML del patrón Singleton
```plantuml
@startuml
class ConfiguracionBanco {
    - _instancia : ConfiguracionBanco
    + nombre_banco
    + tasa_base
    + moneda
    + __new__()
}
ConfiguracionBanco --> ConfiguracionBanco : _instancia (unica)
@enduml
```

## ¿Por qué Singleton y no otro patrón?
- La necesidad concreta es **una única instancia compartida** de configuración, no
  *construir objetos complejos* (Builder) ni *elegir entre familias de productos*
  (Factory/Abstract Factory).
- Podríamos pasar la configuración por parámetro a cada clase, pero eso ensucia todas
  las firmas y no impide que alguien cree otra copia. Singleton **garantiza** la unicidad
  a nivel de la propia clase.
- Es el patrón creacional correcto cuando el problema es **cuántas instancias existen**
  (exactamente una) y no **cómo se construyen**.

> Nota: en sistemas grandes conviene usar Singleton con cuidado (dificulta pruebas). Para
> una configuración global de solo lectura como esta, es un uso legítimo y clásico.